In [1]:
# Prompt: Select 12 geographically distributed representative cooling-priority cells,
# reverse-geocode their centroids, and export Table 4 plus a numbered point layer for Figure 7.

from pathlib import Path
import json
import time
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import geopandas as gpd
import pandas as pd

# مسیر اصلی پروژه
PROJECT_ROOT = Path(r"F:\Isfahan_Urban_Heat_Cooling_Prioritization")

INPUT_GPKG = PROJECT_ROOT / "data" / "processed" / "isfahan_final_cooling_priority_500m.gpkg"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# خواندن لایهٔ نهایی
grid = gpd.read_file(INPUT_GPKG)

# مرکز هندسی هر سلول در EPSG:32639
grid = grid.copy()
grid["center_x"] = grid.geometry.centroid.x
grid["center_y"] = grid.geometry.centroid.y

# تقسیم شهر به چهار پهنهٔ جغرافیایی
mid_x = grid["center_x"].median()
mid_y = grid["center_y"].median()

def assign_sector(row):
    if row["center_x"] < mid_x and row["center_y"] >= mid_y:
        return "شمال‌غرب"
    elif row["center_x"] >= mid_x and row["center_y"] >= mid_y:
        return "شمال‌شرق"
    elif row["center_x"] < mid_x and row["center_y"] < mid_y:
        return "جنوب‌غرب"
    return "جنوب‌شرق"

grid["پهنه_مکانی"] = grid.apply(assign_sector, axis=1)

# انتخاب یک سلول نزدیک به میانهٔ امتیاز هر کلاس در هر پهنه
selected_rows = []

for priority_class in [0, 1, 2]:
    class_data = grid[grid["priority_class"] == priority_class].copy()
    class_median = class_data["final_priority_score"].median()

    for sector in ["شمال‌غرب", "شمال‌شرق", "جنوب‌غرب", "جنوب‌شرق"]:
        sector_data = class_data[class_data["پهنه_مکانی"] == sector].copy()
        selected = sector_data.loc[
            (sector_data["final_priority_score"] - class_median).abs().idxmin()
        ]
        selected_rows.append(selected)

samples = gpd.GeoDataFrame(selected_rows, crs=grid.crs).reset_index(drop=True)

# شماره‌گذاری 1 تا 12
samples["شماره"] = range(1, 13)

priority_labels = {
    0: "اولویت پایین",
    1: "اولویت متوسط",
    2: "اولویت بالا",
}
samples["سطح_اولویت"] = samples["priority_class"].map(priority_labels)

# تبدیل مرکز سلول‌ها به نقطه و سپس به مختصات WGS84
sample_points = gpd.GeoDataFrame(
    samples.drop(columns="geometry"),
    geometry=gpd.points_from_xy(samples["center_x"], samples["center_y"]),
    crs=grid.crs,
)

sample_points_wgs84 = sample_points.to_crs(epsg=4326)
sample_points["عرض_جغرافیایی"] = sample_points_wgs84.geometry.y
sample_points["طول_جغرافیایی"] = sample_points_wgs84.geometry.x

# دریافت نزدیک‌ترین نشانهٔ مکانی از OpenStreetMap / Nominatim
def reverse_geocode(latitude, longitude):
    parameters = urlencode(
        {
            "format": "jsonv2",
            "lat": f"{latitude:.6f}",
            "lon": f"{longitude:.6f}",
            "zoom": 16,
            "addressdetails": 1,
        }
    )

    request = Request(
        f"https://nominatim.openstreetmap.org/reverse?{parameters}",
        headers={
            "User-Agent": (
                "isfahan-urban-heat-cooling-prioritization/"
                "1.0 (GitHub: ParisaGhahreman)"
            )
        },
    )

    try:
        with urlopen(request, timeout=30) as response:
            result = json.loads(response.read().decode("utf-8"))

        address = result.get("address", {})

        road = address.get("road") or address.get("pedestrian") or address.get("footway")
        locality = (
            address.get("neighbourhood")
            or address.get("suburb")
            or address.get("city_district")
            or address.get("village")
        )
        city = address.get("city") or address.get("town") or "اصفهان"

        components = [item for item in [road, locality, city] if item]
        return "، ".join(dict.fromkeys(components)) if components else result.get("display_name", "یافت نشد")

    except Exception as error:
        return f"نیازمند بررسی دستی ({type(error).__name__})"

addresses = []

for _, row in sample_points.iterrows():
    addresses.append(
        reverse_geocode(row["عرض_جغرافیایی"], row["طول_جغرافیایی"])
    )
    time.sleep(1.1)  # رعایت محدودیت سرویس OpenStreetMap

sample_points["نزدیک‌ترین_نشانه_مکانی"] = addresses

# آماده‌سازی جدول 4
table_04 = sample_points[
    [
        "شماره",
        "grid_id",
        "سطح_اولویت",
        "final_priority_score",
        "پهنه_مکانی",
        "عرض_جغرافیایی",
        "طول_جغرافیایی",
        "نزدیک‌ترین_نشانه_مکانی",
    ]
].copy()

table_04 = table_04.rename(
    columns={
        "grid_id": "شناسه_سلول",
        "final_priority_score": "امتیاز_نهایی",
    }
)

table_04["امتیاز_نهایی"] = table_04["امتیاز_نهایی"].round(4)
table_04["عرض_جغرافیایی"] = table_04["عرض_جغرافیایی"].round(6)
table_04["طول_جغرافیایی"] = table_04["طول_جغرافیایی"].round(6)

# ذخیرهٔ جدول مقاله
table_04.to_csv(
    TABLES_DIR / "table_04_representative_cells.csv",
    index=False,
    encoding="utf-8-sig",
)

table_04.to_excel(
    TABLES_DIR / "table_04_representative_cells.xlsx",
    index=False,
)

# ذخیرهٔ نقطه‌های شماره‌دار برای افزودن به شکل 7 در QGIS
point_layer = sample_points[
    [
        "شماره",
        "grid_id",
        "سطح_اولویت",
        "final_priority_score",
        "پهنه_مکانی",
        "عرض_جغرافیایی",
        "طول_جغرافیایی",
        "نزدیک‌ترین_نشانه_مکانی",
        "geometry",
    ]
].copy()

point_layer.to_file(
    PROCESSED_DIR / "isfahan_representative_cells_12.gpkg",
    layer="representative_cells_12",
    driver="GPKG",
)

display(table_04)
print("Table 4 saved to:", TABLES_DIR / "table_04_representative_cells.xlsx")
print("Point layer saved to:", PROCESSED_DIR / "isfahan_representative_cells_12.gpkg")

,شماره,شناسه_سلول,سطح_اولویت,امتیاز_نهایی,پهنه_مکانی,عرض_جغرافیایی,طول_جغرافیایی,نزدیک‌ترین_نشانه_مکانی
0,1,ISF_0708,اولویت پایین,0.0343,شمال‌غرب,32.766802,51.630830,آزادراه معلم، منطقه ۱۲، اصفهان
1,2,ISF_1947,اولویت پایین,0.0335,شمال‌شرق,32.685989,51.786695,منطقه ۱۵، اصفهان
2,3,ISF_0836,اولویت پایین,0.0358,جنوب‌غرب,32.592069,51.647350,منطقه ۵، اصفهان
3,4,ISF_2179,اولویت پایین,0.0349,جنوب‌شرق,32.640615,51.828943,کاج، منطقه ۴، اصفهان
4,5,ISF_0382,اولویت متوسط,0.3010,شمال‌غرب,32.673490,51.599942,بهشت، زاجان، اصفهان
5,6,ISF_1501,اولویت متوسط,0.3010,شمال‌شرق,32.654834,51.717112,ساحل، همدانیان، اصفهان
6,7,ISF_0736,اولویت متوسط,0.3010,جنوب‌غرب,32.646246,51.637077,مهدی فرقدانی، محدوده ناژوان منطقه ۹، اصفهان
7,8,ISF_1173,اولویت متوسط,0.3010,جنوب‌شرق,32.637002,51.679656,کمال اسماعیل، چرخاب، اصفهان
8,9,ISF_0461,اولویت بالا,0.3880,شمال‌غرب,32.790728,51.606067,آزادراه معلم، منطقه ۱۲، اصفهان
9,10,ISF_1241,اولویت بالا,0.3862,شمال‌شرق,32.727175,51.685676,بهاران ۱ شرقی، شهرک میلاد، اصفهان


Table 4 saved to: F:\Isfahan_Urban_Heat_Cooling_Prioritization\outputs\tables\table_04_representative_cells.xlsx
Point layer saved to: F:\Isfahan_Urban_Heat_Cooling_Prioritization\data\processed\isfahan_representative_cells_12.gpkg


In [3]:
# Prompt: Select 12 spatially clear core cells, distributed across Isfahan,
# reverse-geocode them, and overwrite Table 4 and the numbered Figure 7 point layer.

from pathlib import Path
import json
import time
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import geopandas as gpd
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"F:\Isfahan_Urban_Heat_Cooling_Prioritization")

INPUT_GPKG = PROJECT_ROOT / "data" / "processed" / "isfahan_final_cooling_priority_500m.gpkg"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

grid = gpd.read_file(INPUT_GPKG).copy()

# مختصات مرکز هر سلول در واحد متر
grid["center_x"] = grid.geometry.centroid.x
grid["center_y"] = grid.geometry.centroid.y

priority_labels = {
    0: "اولویت پایین",
    1: "اولویت متوسط",
    2: "اولویت بالا",
}

def select_spatially_clear_cells(class_data, n_samples=4, radius_m=1500, min_separation_m=3000):
    """
    انتخاب سلول‌های هسته‌ای هر کلاس بر اساس بیشترین تراکم همسایگیِ هم‌کلاس،
    همراه با حداقل فاصلهٔ مکانی میان نمونه‌ها.
    """
    class_data = class_data.copy().reset_index(drop=True)

    coords = class_data[["center_x", "center_y"]].to_numpy()
    dx = coords[:, 0][:, None] - coords[:, 0][None, :]
    dy = coords[:, 1][:, None] - coords[:, 1][None, :]
    distances = np.sqrt(dx**2 + dy**2)

    # تعداد سلول‌های هم‌کلاس در شعاع 1.5 کیلومتری؛ معیار هسته‌بودن پهنه
    class_data["local_class_density"] = (distances <= radius_m).sum(axis=1)

    median_score = class_data["final_priority_score"].median()

    ranked_indices = sorted(
        class_data.index,
        key=lambda idx: (
            -class_data.loc[idx, "local_class_density"],
            abs(class_data.loc[idx, "final_priority_score"] - median_score),
        ),
    )

    chosen_indices = []
    active_min_separation = min_separation_m

    # اگر چهار نقطه با فاصلهٔ 3 کیلومتر ممکن نبود، فاصله به‌تدریج کاهش می‌یابد.
    while len(chosen_indices) < n_samples and active_min_separation >= 1000:
        for idx in ranked_indices:
            if idx in chosen_indices:
                continue

            if not chosen_indices:
                chosen_indices.append(idx)
            else:
                candidate = coords[idx]
                chosen_coords = coords[chosen_indices]
                minimum_distance = np.sqrt(
                    ((chosen_coords - candidate) ** 2).sum(axis=1)
                ).min()

                if minimum_distance >= active_min_separation:
                    chosen_indices.append(idx)

            if len(chosen_indices) == n_samples:
                break

        active_min_separation -= 500

    return class_data.loc[chosen_indices].copy()

selected_groups = []

for priority_class in [0, 1, 2]:
    class_data = grid[grid["priority_class"] == priority_class].copy()
    selected = select_spatially_clear_cells(class_data, n_samples=4)
    selected["سطح_اولویت"] = priority_labels[priority_class]
    selected_groups.append(selected)

samples = pd.concat(selected_groups, ignore_index=True)
samples["شماره"] = range(1, 13)

# ساخت نقطهٔ مرکز سلول‌ها
sample_points = gpd.GeoDataFrame(
    samples.drop(columns="geometry"),
    geometry=gpd.points_from_xy(samples["center_x"], samples["center_y"]),
    crs=grid.crs,
)

# تبدیل به مختصات جغرافیایی WGS84
points_wgs84 = sample_points.to_crs(epsg=4326)
sample_points["عرض_جغرافیایی"] = points_wgs84.geometry.y
sample_points["طول_جغرافیایی"] = points_wgs84.geometry.x

def reverse_geocode(latitude, longitude):
    parameters = urlencode(
        {
            "format": "jsonv2",
            "lat": f"{latitude:.6f}",
            "lon": f"{longitude:.6f}",
            "zoom": 16,
            "addressdetails": 1,
        }
    )

    request = Request(
        f"https://nominatim.openstreetmap.org/reverse?{parameters}",
        headers={
            "User-Agent": (
                "isfahan-urban-heat-cooling-prioritization/"
                "1.0 (GitHub: ParisaGhahreman)"
            )
        },
    )

    try:
        with urlopen(request, timeout=30) as response:
            result = json.loads(response.read().decode("utf-8"))

        return result.get("display_name", "یافت نشد")

    except Exception as error:
        return f"نیازمند بررسی دستی ({type(error).__name__})"

addresses = []

for _, row in sample_points.iterrows():
    addresses.append(
        reverse_geocode(row["عرض_جغرافیایی"], row["طول_جغرافیایی"])
    )
    time.sleep(1.1)

sample_points["نزدیک‌ترین_نشانه_مکانی"] = addresses

# جدول 4
table_04 = sample_points[
    [
        "شماره",
        "grid_id",
        "سطح_اولویت",
        "final_priority_score",
        "local_class_density",
        "عرض_جغرافیایی",
        "طول_جغرافیایی",
        "نزدیک‌ترین_نشانه_مکانی",
    ]
].copy()

table_04 = table_04.rename(
    columns={
        "grid_id": "شناسه_سلول",
        "final_priority_score": "امتیاز_نهایی",
        "local_class_density": "تراکم_محلی_هم‌کلاس",
    }
)

table_04["امتیاز_نهایی"] = table_04["امتیاز_نهایی"].round(4)
table_04["عرض_جغرافیایی"] = table_04["عرض_جغرافیایی"].round(6)
table_04["طول_جغرافیایی"] = table_04["طول_جغرافیایی"].round(6)

table_04.to_csv(
    TABLES_DIR / "table_04_representative_cells.csv",
    index=False,
    encoding="utf-8-sig",
)

table_04.to_excel(
    TABLES_DIR / "table_04_representative_cells.xlsx",
    index=False,
)

# لایهٔ نقطه‌ای شماره‌دار برای شکل 7
point_layer = sample_points[
    [
        "شماره",
        "grid_id",
        "سطح_اولویت",
        "final_priority_score",
        "local_class_density",
        "عرض_جغرافیایی",
        "طول_جغرافیایی",
        "نزدیک‌ترین_نشانه_مکانی",
        "geometry",
    ]
].copy()

point_layer.to_file(
    PROCESSED_DIR / "isfahan_representative_cells_12.gpkg",
    layer="representative_cells_12",
    driver="GPKG",
)

display(table_04)
print("Updated Table 4:", TABLES_DIR / "table_04_representative_cells.xlsx")
print("Updated point layer:", PROCESSED_DIR / "isfahan_representative_cells_12.gpkg")

,شماره,شناسه_سلول,سطح_اولویت,امتیاز_نهایی,تراکم_محلی_هم‌کلاس,عرض_جغرافیایی,طول_جغرافیایی,نزدیک‌ترین_نشانه_مکانی
0,1,ISF_0228,اولویت پایین,0.0019,30,32.646505,51.583768,"آل محمد, باغ برج, نصرآباد, منطقه ۹, اصفهان, بخ..."
1,2,ISF_1781,اولویت پایین,0.0082,29,32.609483,51.759382,"شهید سید اکبریان, پینارت, منطقه ۴, روشن دشت, ا..."
2,3,ISF_0602,اولویت پایین,0.0010,29,32.750059,51.621804,"شهرک نمایشگاه, منطقه ۱۲, اصفهان, بخش مرکزی شهر..."
3,4,ISF_1853,اولویت پایین,0.0299,28,32.672559,51.770580,"رویان, شهرک امام حسین, خوراسگان, منطقه ۱۵, اصف..."
4,5,ISF_0736,اولویت متوسط,0.3010,29,32.646246,51.637077,"مهدی فرقدانی, محدوده ناژوان منطقه ۹, منطقه ۹, ..."
5,6,ISF_0434,اولویت متوسط,0.3000,29,32.668955,51.605244,"ایمان, گلستان - سودان, منطقه ۹, اصفهان, بخش مر..."
6,7,ISF_1366,اولویت متوسط,0.3000,29,32.645905,51.701048,"برادران, فرهنگیان, منطقه ۴, اصفهان, بخش مرکزی ..."
7,8,ISF_1070,اولویت متوسط,0.3019,27,32.641569,51.669029,"ماه, باغ نگار - آیینه خانه, منطقه ۶, اصفهان, ب..."
8,9,ISF_1564,اولویت بالا,0.6743,30,32.569081,51.727082,"بزرگراه شهید علی بالایی, منطقه ۶, سپاهان شهر, ..."
9,10,ISF_0826,اولویت بالا,0.5818,29,32.546967,51.647026,"منطقه ۵, اصفهان, بخش مرکزی شهرستان اصفهان, شهر..."


Updated Table 4: F:\Isfahan_Urban_Heat_Cooling_Prioritization\outputs\tables\table_04_representative_cells.xlsx
Updated point layer: F:\Isfahan_Urban_Heat_Cooling_Prioritization\data\processed\isfahan_representative_cells_12.gpkg
